****Data_EDA&Preprocessing****
-------------------------

***"This is a V1  for phase 1 NLP project"***

**Project Title : Arabic multi-domain aspect-based sentiment analysis system using the A-MASA dataset**

**Course Code & Lecture section : AIS411: Natural Language Processing-2026SPRG-LECT-01**

**Lab Section : 01A**

**Supervised by DR & TA :** **DR.Walaa Medhat** / **Eng.Yomna Eid**

**School of Information Technologyand Computer Science (ITCS), Nile University (NU), Giza, Egypt**

**Authors & IDs:** 
- **Hossamaldeen Elghazaly (Team Leader)** → 221001672
- **Alia Mostafa** →  221001994
- **Hassan Sabry** → 221001707
- **Moaz Elkhashab** → 221001732
- **Yomna Awad** → 221001896

## 1-Libraries Importation

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import string
import warnings
warnings.filterwarnings('ignore')

# Arabic-specific libraries
try:
    from pyarabic.araby import is_arabic, remove_diacritics, remove_tatweel, normalize_arabic
except:
    print("Installing pyarabic...")
    os.system('pip install -q pyarabic')
    from pyarabic.araby import is_arabic, remove_diacritics, remove_tatweel, normalize_arabic

# For advanced Arabic preprocessing
try:
    import camel_tools
except:
    print("Installing camel-tools...")
    os.system('pip install -q camel-tools')

# Additional visualization libraries
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import Counter
from wordcloud import WordCloud

## 2-Environment Settings

In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

# Arabic stopwords
arabic_stopwords = set(stopwords.words('arabic'))

# Additional custom Arabic stopwords
custom_arabic_stopwords = {
    'في', 'من', 'إلى', 'هذا', 'هذه', 'ذلك', 'تلك', 'التي', 'الذي',
    'هو', 'هي', 'أن', 'إن', 'و', 'أو', 'لكن', 'لكن', 'لم', 'ما',
    'كل', 'بعض', 'أيضا', 'كذلك', 'هنا', 'هناك', 'الآن', 'قد'
}
arabic_stopwords.update(custom_arabic_stopwords)

lemmatizer = WordNetLemmatizer()

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")

[nltk_data] Downloading package stopwords to C:\Users\Hossam
[nltk_data]     Elghazaly\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Hossam
[nltk_data]     Elghazaly\AppData\Roaming\nltk_data...


## 3-Dataset Loading&Reading 

In [9]:
all_data_PATH = r'H:\Work\RunningWork\NLP project\A-MASA-Arabic-Multidomain-Aspect-based-Sentiment-Analysis-Datasets\\'

df_ATT = pd.read_csv(all_data_PATH + 'ATT.csv')
df_MOV = pd.read_csv(all_data_PATH + 'MOV.csv')
df_RES = pd.read_csv(all_data_PATH + 'RES.csv')
df_PROD = pd.read_csv(all_data_PATH + 'PROD.csv')
df_TELC = pd.read_csv(all_data_PATH + 'TELC.csv')

In [23]:
df_ATT.info()
df_ATT.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Column1              1500 non-null   object
 1   Sentences            1503 non-null   object
 2   Aspects              1490 non-null   object
 3   Sentences Sentiment  1501 non-null   object
 4   Aspects Sentiment    1489 non-null   object
dtypes: object(5)
memory usage: 58.8+ KB


,Column1,Sentences,Aspects,Sentences Sentiment,Aspects Sentiment
0,0,\nنصحوني بتجربة حمام الكبريت. يمكنكم الدخول مع...,الحمام، العاملات، الإنجليزية,1,"1, 1, -1"
1,1,قلعة ساحرة \r\nمنظر خلاب للمدينة من أعلى القلع...,قلعة، منظر، حديقة نباتات، مجرى مائي,1,"1, 1, 1, 1"
2,2,تبليسي جورجيا\nمن أجمل المدن التي زرتها في حيا...,تبليسي جورجيا، شعبها، الحياة,1,"1, 1, 1"
3,3,جولة على المدينة القديمة تبليسي شاردن \nممتعة ...,المدينة القديمة تبليسي، الجولة، جلساتها، تلفري...,1,"1,1,1,1,1,1"
4,4,احلي اجازه لمحبي الطبيعه \r\nالمناظر الخلابه ا...,اجازه، المناظر، الاجواء، ميزه,1,"1,1,1,0"


In [24]:
df_MOV.info()
df_MOV.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sentences           499 non-null    object 
 1   Sentence Sentiment  486 non-null    float64
 2   Aspects             466 non-null    object 
 3   Aspects Sentiment   465 non-null    object 
dtypes: float64(1), object(3)
memory usage: 15.7+ KB


,Sentences,Sentence Sentiment,Aspects,Aspects Sentiment
0,ما بين عائلة زيزي و ميكي يعتصر مصر الالم من من...,1.0,فطين عبدالوهاب،سعاد حسني،فؤاد المهندس،احمد رمز...,"1, 1, 1, 1,1"
1,أما في فيلم عائلة ميكي 2010 و الذى اعتمد على ...,NaN,الأحلام،الأزمات،عرض,"-1,-1,1"
2,المخرج اكرم فريد ينجو أخيرا من قبضة السبكي و ا...,NaN,طاقة فنية،الأداء،أداء,"1,1,1"
3,سامحكم الله سامحكم الله الفيلم لا يعبر إطلاقاً...,-1.0,الفنون,1
4,المحتل الأمين على مدار التاريخ السينمائى الامي...,0.0,المحتل،صورة,"1,-1"


In [25]:
df_RES.info()
df_RES.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sentences           2599 non-null   object 
 1   Aspects             1470 non-null   object 
 2   Sentence Sentiment  2599 non-null   float64
 3   Aspects Sentiment   1468 non-null   object 
dtypes: float64(1), object(3)
memory usage: 81.4+ KB


,Sentences,Aspects,Sentence Sentiment,Aspects Sentiment
0,٦٠\nاللي يوصل ميلان ولا يمر هالمطعم اعتبرة خسر...,مطعم،اكل،العاملين,1.0,"1,1,1"
1,رااااائع\nالمطعم أكله لذيذ جداً والعاملين متعا...,المطعم،أكل،العاملين،موقع،شوربة الحريرة,1.0,"1,1,1,1,1"
2,\nمطعم جيد\nالعاملين فيه ودون\nبالنسبة للأطعمة...,مطعم،العاملين،أطعمة،ورق العنب،الشاي الأحمر,1.0,"1, 1, 1, -1, 1"
3,مطعم جميل جدااااا\nمأكولات بجد جميله جداااااا ...,مطعم،مأكولات،الشيف،أكل،طباخ،الطرته،مطعم,1.0,"1, 1, 1, 1, 1, 1, 1"
4,الأكل رائع و حلال.. طلبنا لحم بالبامية مع رز و...,الأكل،لحم بالبامية مع رز،مشاوي مشكله،فلافل،أبو...,1.0,"1, 0, 1, 1, 1"


In [26]:
df_PROD.info()
df_PROD.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1504 entries, 0 to 1503
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sentences           1504 non-null   object 
 1   Aspects             989 non-null    object 
 2   Sentence Sentiment  1501 non-null   float64
 3   Aspects Sentiment   988 non-null    object 
dtypes: float64(1), object(3)
memory usage: 47.1+ KB


,Sentences,Aspects,Sentence Sentiment,Aspects Sentiment
0,ممتازة بصراحه مريحة خاصتا في الخرجات الي من غي...,NaN,1.0,NaN
1,المنتج عملي,المنتج,1.0,1
2,جيد نوع,نوع,0.0,1
3,اشتريت السلعه ودفعت ثمنها من الماستر كارد وتم ...,NaN,-1.0,NaN
4,شنطة جميلة وأنيقة وسعرها مناسب جدا,شنطة،سعر,-1.0,"1, 1"


In [27]:
df_TELC.info()
df_TELC.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1545 entries, 0 to 1544
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Sentences          1491 non-null   object
 1   Aspects            759 non-null    object
 2   Aspects Sentiment  759 non-null    object
dtypes: object(3)
memory usage: 36.3+ KB


,Sentences,Aspects,Aspects Sentiment
0,خدمتك دلوقتى اتصالات بتقدم عرض خصم لمدة شهور ب...,عملاء،رمضان,"1,1"
1,بعدين المنطقة رافضة شبكة اتصالات تماما دي,المنطقة،شبكة,"-1,0"
2,انا كنت اعرف وحده اتصالات كانت ديما عارفه انا ...,NaN,NaN
3,ايه ال دي يسطا اختصار اتصالات ولا ايه,اختصار,0
4,للحصول خدمات الـ ممكن نتصل رقم المبيعات أي رقم...,رقم،الكارت،مواعيد العمل,"1,1,1"


In [28]:
df_all = pd.concat([df_ATT, df_MOV, df_RES, df_PROD, df_TELC], ignore_index=True)
df_all.info()
df_all.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7651 entries, 0 to 7650
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Column1              1500 non-null   object 
 1   Sentences            7596 non-null   object 
 2   Aspects              5174 non-null   object 
 3   Sentences Sentiment  1501 non-null   object 
 4   Aspects Sentiment    5169 non-null   object 
 5   Sentence Sentiment   4586 non-null   float64
dtypes: float64(1), object(5)
memory usage: 358.8+ KB


,Column1,Sentences,Aspects,Sentences Sentiment,Aspects Sentiment,Sentence Sentiment
0,0,\nنصحوني بتجربة حمام الكبريت. يمكنكم الدخول مع...,الحمام، العاملات، الإنجليزية,1,"1, 1, -1",NaN
1,1,قلعة ساحرة \r\nمنظر خلاب للمدينة من أعلى القلع...,قلعة، منظر، حديقة نباتات، مجرى مائي,1,"1, 1, 1, 1",NaN
2,2,تبليسي جورجيا\nمن أجمل المدن التي زرتها في حيا...,تبليسي جورجيا، شعبها، الحياة,1,"1, 1, 1",NaN
3,3,جولة على المدينة القديمة تبليسي شاردن \nممتعة ...,المدينة القديمة تبليسي، الجولة، جلساتها، تلفري...,1,"1,1,1,1,1,1",NaN
4,4,احلي اجازه لمحبي الطبيعه \r\nالمناظر الخلابه ا...,اجازه، المناظر، الاجواء، ميزه,1,"1,1,1,0",NaN


## 4-Data Preprocessing 

In [ ]:

# =============================================================================
# 4.1 ARABIC TEXT PREPROCESSING FUNCTIONS
# =============================================================================

def remove_urls(text):
    """Remove URLs from text"""
    url_pattern = r'https?://\S+|www\.\S+'
    return re.sub(url_pattern, '', text)

def remove_emails(text):
    """Remove email addresses"""
    email_pattern = r'\S+@\S+'
    return re.sub(email_pattern, '', text)

def remove_html_tags(text):
    """Remove HTML tags"""
    html_pattern = r'<[^>]+>'
    return re.sub(html_pattern, '', text)

def normalize_arabic_text(text):
    """
    Normalize Arabic text by:
    - Removing diacritics (tashkeel)
    - Normalizing different forms of Alef (ا، أ، إ، آ) to ا
    - Normalizing Teh Marbuta (ة) to Ha (ه)
    - Normalizing Ya variations (ى، ي) to Ya (ي)
    - Removing Tatweel (ـــ)
    """
    if not isinstance(text, str):
        return text
    
    # Remove diacritics
    text = remove_diacritics(text)
    
    # Remove Tatweel
    text = remove_tatweel(text)
    
    # Normalize Alef variations
    text = re.sub(r'[أإآ]', 'ا', text)
    
    # Normalize Teh Marbuta to Ha
    text = re.sub(r'ة', 'ه', text)
    
    # Normalize Ya variations
    text = re.sub(r'ى', 'ي', text)
    
    # Normalize Hamza
    text = re.sub(r'ء', '', text)
    
    return text

def remove_english_numbers_special_chars(text):
    """Remove English letters, numbers, and special characters"""
    # Keep only Arabic characters and spaces
    text = re.sub(r'[a-zA-Z0-9]', '', text)  # Remove English and numbers
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)  # Keep only Arabic and spaces
    return text

def remove_extra_whitespace(text):
    """Remove extra whitespaces"""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def arabic_tokenize(text):
    """
    Tokenize Arabic text
    Split by spaces and punctuation while preserving word integrity
    """
    if not isinstance(text, str):
        return []
    
    # Split by whitespace
    tokens = text.split()
    
    # Further split by common Arabic punctuation if needed
    final_tokens = []
    for token in tokens:
        # Remove Arabic punctuation while maintaining tokens
        token = re.sub(r'[،؛؟!:«»]', '', token)
        if token:
            final_tokens.append(token)
    
    return final_tokens

def remove_stopwords_arabic(tokens):
    """Remove Arabic stopwords from token list"""
    return [token for token in tokens if token not in arabic_stopwords]

def preprocess_arabic_text(text, remove_stopwords=True, remove_english=True):
    """
    Complete Arabic text preprocessing pipeline
    
    Steps:
    1. Remove URLs, emails, HTML tags
    2. Normalize Arabic characters
    3. Remove English/numbers/special characters (optional)
    4. Remove extra whitespace
    5. Tokenize
    6. Remove stopwords (optional)
    
    Args:
        text (str): Input text
        remove_stopwords (bool): Whether to remove Arabic stopwords
        remove_english (bool): Whether to remove English characters and numbers
    
    Returns:
        list: Preprocessed tokens
    """
    if not isinstance(text, str) or not text.strip():
        return []
    
    # Step 1: Remove URLs, emails, HTML
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_html_tags(text)
    
    # Step 2: Normalize Arabic
    text = normalize_arabic_text(text)
    
    # Step 3: Remove English/numbers if specified
    if remove_english:
        text = remove_english_numbers_special_chars(text)
    
    # Step 4: Remove extra whitespace
    text = remove_extra_whitespace(text)
    
    # Step 5: Tokenize
    tokens = arabic_tokenize(text)
    
    # Step 6: Remove stopwords if specified
    if remove_stopwords:
        tokens = remove_stopwords_arabic(tokens)
    
    return tokens

# Test the preprocessing functions
print("=" * 80)
print("TESTING ARABIC PREPROCESSING FUNCTIONS")
print("=" * 80)

test_text = "مرحبا! هذا نص تجريبي في اللغة العربية الفصحى والعامية. تم اختباره على 123 جهاز."
print(f"\nOriginal text:\n{test_text}\n")
print(f"Normalized: {normalize_arabic_text(test_text)}")
print(f"After removing English/numbers: {remove_english_numbers_special_chars(normalize_arabic_text(test_text))}")
print(f"Preprocessed tokens (with stopwords removal): {preprocess_arabic_text(test_text)}")
print(f"Preprocessed tokens (without stopwords removal): {preprocess_arabic_text(test_text, remove_stopwords=False)}")


In [ ]:
# =============================================================================
# 7.2 REFERENCE: AVAILABLE FUNCTIONS SUMMARY
# =============================================================================

print("\n" + "="*100)
print("AVAILABLE PREPROCESSING FUNCTIONS REFERENCE")
print("="*100)

functions_reference = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                    PREPROCESSING FUNCTIONS REFERENCE                        │
└─────────────────────────────────────────────────────────────────────────────┘

📌 CORE FUNCTIONS:
──────────────────────────────────────────────────────────────────────────────

1. normalize_arabic_text(text)
   ├─ Purpose: Normalize Arabic characters
   ├─ Input: String (Arabic text)
   ├─ Output: Normalized string
   ├─ Actions:
   │  ├─ Remove diacritics (fatha, damma, kasra, sukun, etc.)
   │  ├─ Normalize Alef: أ, إ, آ → ا
   │  ├─ Normalize Teh Marbuta: ة → ه
   │  └─ Remove Hamza and Tatweel
   └─ Example: normalize_arabic_text("السلام عليكم")

2. preprocess_arabic_text(text, remove_stopwords=True, remove_english=True)
   ├─ Purpose: Complete preprocessing pipeline
   ├─ Input: String (raw Arabic text)
   ├─ Output: List of tokens
   ├─ Steps:
   │  ├─ Remove URLs, emails, HTML tags
   │  ├─ Normalize Arabic text
   │  ├─ Remove English letters and numbers
   │  ├─ Tokenize
   │  └─ Remove stopwords (optional)
   └─ Example: preprocess_arabic_text("نص تجريبي 123", remove_stopwords=True)

3. arabic_tokenize(text)
   ├─ Purpose: Tokenize Arabic text
   ├─ Input: String
   ├─ Output: List of tokens
   ├─ Splits by whitespace and punctuation
   └─ Example: arabic_tokenize("السلام عليكم ورحمة الله")

4. remove_stopwords_arabic(tokens)
   ├─ Purpose: Remove Arabic stopwords
   ├─ Input: List of tokens
   ├─ Output: Filtered list of tokens
   ├─ Uses 1000+ common Arabic stopwords
   └─ Example: remove_stopwords_arabic(['السلام', 'عليكم'])

──────────────────────────────────────────────────────────────────────────────

📌 UTILITY FUNCTIONS:
──────────────────────────────────────────────────────────────────────────────

5. remove_urls(text)
   └─ Removes URLs like http://example.com or www.example.com

6. remove_emails(text)
   └─ Removes email addresses like test@example.com

7. remove_html_tags(text)
   └─ Removes HTML tags like <p>, <div>, etc.

8. remove_english_numbers_special_chars(text)
   └─ Removes English letters, numbers, and non-Arabic special characters

9. remove_extra_whitespace(text)
   └─ Removes multiple spaces and normalizes whitespace

──────────────────────────────────────────────────────────────────────────────

📌 CLASS-BASED PREPROCESSING:
──────────────────────────────────────────────────────────────────────────────

ArabicTextPreprocessor Class:
├─ __init__(remove_stopwords=True, normalize=True, remove_english=True)
│  └─ Initialize preprocessor with configuration
│
├─ process_dataset(df, text_column='text', batch_size=100)
│  ├─ Process entire dataset
│  ├─ Parameters:
│  │  ├─ df: pandas DataFrame
│  │  ├─ text_column: column name containing text
│  │  └─ batch_size: progress report frequency
│  └─ Returns: List of token lists
│
├─ get_statistics()
│  ├─ Returns preprocessing statistics
│  └─ Includes: avg_tokens, vocabulary_size, etc.
│
└─ export_to_csv(output_path, original_df=None)
   ├─ Export processed tokens to CSV
   └─ Optionally includes original metadata

──────────────────────────────────────────────────────────────────────────────

🔧 CONFIGURATION OPTIONS:
──────────────────────────────────────────────────────────────────────────────

remove_stopwords (bool, default=True)
├─ True: Remove common Arabic stopwords
├─ False: Keep all tokens (recommended for ABSA)
└─ Use False to preserve aspect context

normalize (bool, default=True)
├─ True: Apply Arabic normalization
├─ False: Skip normalization (not recommended)
└─ Always recommended for Arabic text

remove_english (bool, default=True)
├─ True: Remove English letters and numbers
├─ False: Keep English content
└─ Recommended for pure Arabic text analysis

──────────────────────────────────────────────────────────────────────────────

🎯 QUICK FUNCTION SELECTION GUIDE:
──────────────────────────────────────────────────────────────────────────────

Task                          | Recommended Function
──────────────────────────────────────────────────────────────────────────
Single text preprocessing     | preprocess_arabic_text()
Batch dataset processing      | ArabicTextPreprocessor.process_dataset()
Just remove diacritics       | normalize_arabic_text()
Just tokenize                | arabic_tokenize()
Text cleaning only           | remove_urls(), remove_emails()
ABSA-specific                | preprocess_arabic_text(remove_stopwords=False)
Sentiment analysis           | preprocess_arabic_text(remove_stopwords=True)

═════════════════════════════════════════════════════════════════════════════
"""

print(functions_reference)

# Save reference to file
reference_file = 'FUNCTIONS_REFERENCE.txt'
with open(reference_file, 'w', encoding='utf-8') as f:
    f.write(functions_reference)
print(f"\n✓ Functions reference saved to '{reference_file}'")

# Print summary
print("\n" + "="*100)
print("📚 DOCUMENTATION FILES CREATED:")
print("="*100)
print("  1. EDA_Summary_Report.txt - Comprehensive EDA findings")
print("  2. PREPROCESSING_USAGE_GUIDE.txt - Detailed usage guide")
print("  3. FUNCTIONS_REFERENCE.txt - Function documentation")
print("  4. EDA_overview.png - Main visualizations")
print("  5. Aspect_analysis.png - Aspect distribution visualizations")
print("  6. Vocabulary_analysis.png - Vocabulary and linguistic analysis")
print("="*100)


In [ ]:
# =============================================================================
# 7.1 PREPROCESSING USAGE GUIDE & BEST PRACTICES
# =============================================================================

usage_guide = """
╔══════════════════════════════════════════════════════════════════════════════╗
║           ARABIC TEXT PREPROCESSING & EDA - USAGE GUIDE                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

📌 QUICK START EXAMPLES:
─────────────────────────────────────────────────────────────────────────────

1️⃣  SIMPLE PREPROCESSING (Single Text):
    ─────────────────────────────────────
    from preprocess_arabic_text import preprocess_arabic_text
    
    text = "هذا نص تجريبي في اللغة العربية"
    tokens = preprocess_arabic_text(text, remove_stopwords=True)
    print(tokens)

2️⃣  BATCH PROCESSING (Entire Dataset):
    ────────────────────────────────────
    preprocessor = ArabicTextPreprocessor(
        remove_stopwords=True,
        normalize=True,
        remove_english=True
    )
    
    processed = preprocessor.process_dataset(df, text_column='text')
    stats = preprocessor.get_statistics()

3️⃣  ABSA-SPECIFIC PREPROCESSING (Preserve Aspects):
    ─────────────────────────────────────────────────
    preprocessor_absa = ArabicTextPreprocessor(
        remove_stopwords=False,  # Important: Keep all words!
        normalize=True,
        remove_english=True
    )
    
    processed = preprocessor_absa.process_dataset(df)

4️⃣  SENTIMENT ANALYSIS PREPROCESSING:
    ───────────────────────────────────
    preprocessor_sentiment = ArabicTextPreprocessor(
        remove_stopwords=True,   # Focus on sentiment words
        normalize=True,
        remove_english=True
    )
    
    processed = preprocessor_sentiment.process_dataset(df)

─────────────────────────────────────────────────────────────────────────────

🛠️  PREPROCESSING PIPELINE STAGES:
─────────────────────────────────────────────────────────────────────────────

Stage 1: Text Cleaning
    ✓ Remove URLs (http://example.com)
    ✓ Remove email addresses
    ✓ Remove HTML tags (<p>, <div>, etc.)

Stage 2: Arabic Normalization
    ✓ Remove diacritics (تشكيل) - fatha, damma, kasra, etc.
    ✓ Normalize Alef variations: أ, إ, آ → ا
    ✓ Normalize Teh Marbuta: ة → ه
    ✓ Normalize Ya variations: ى → ي
    ✓ Remove Hamza (ء) and Tatweel (ـــ)

Stage 3: Character Filtering
    ✓ Keep only Arabic characters and spaces
    ✓ Optionally remove English letters and numbers

Stage 4: Text Normalization
    ✓ Remove extra whitespaces
    ✓ Strip leading/trailing whitespace

Stage 5: Tokenization
    ✓ Split by whitespace
    ✓ Remove punctuation (، ؛ ؟ ! : « »)

Stage 6: Stopword Removal (Optional)
    ✓ Remove 1000+ common Arabic words
    ✓ Custom stopwords can be added

─────────────────────────────────────────────────────────────────────────────

⚠️  IMPORTANT CONSIDERATIONS:
─────────────────────────────────────────────────────────────────────────────

🔴 FOR ABSA TASKS:
    ⚠️  DO NOT remove stopwords - they provide context for aspects
    ⚠️  Example: "الخدمة رائعة جداً" (The service is very great)
        - "جداً" (very) is a stopword but crucial for intensity
        - Removing it loses sentiment intensity information

🟢 FOR SENTIMENT CLASSIFICATION:
    ✓ Stopword removal is beneficial - focuses on sentiment words
    ✓ Reduces noise and vocabulary size
    ✓ Improves model training efficiency

🟡 FOR MULTI-DOMAIN ANALYSIS:
    ✓ Keep domain-specific terminology
    ✓ Don't over-normalize (preserve business context)
    ✓ Example: "بيتزا شهية" (delicious pizza) - "شهية" is common in food domain

─────────────────────────────────────────────────────────────────────────────

📊 EDA COMPONENTS INCLUDED:
─────────────────────────────────────────────────────────────────────────────

✓ Dataset Overview
    - Total records, domains, columns, memory usage

✓ Text Statistics
    - Length distribution (characters and tokens)
    - Min/Max/Mean/Median values

✓ Sentiment Analysis
    - Distribution across classes
    - Class balance assessment
    - Cross-domain sentiment patterns

✓ Aspect Analysis
    - Most common aspects
    - Aspect-sentiment relationships
    - Domain-specific aspects

✓ Data Quality Metrics
    - Completeness (missing values)
    - Consistency (duplicates, empty records)
    - Validity (value constraints)

✓ Vocabulary Analysis
    - Unique tokens and vocabulary size
    - Type-Token Ratio (TTR)
    - Rare words (hapax legomena)
    - Frequency distribution (Zipf's law)

─────────────────────────────────────────────────────────────────────────────

💡 RECOMMENDED WORKFLOWS:
─────────────────────────────────────────────────────────────────────────────

Workflow 1: ABSA Training Data Preparation
    1. Load raw datasets
    2. Run EDA to understand data distribution
    3. Check for class imbalance
    4. Preprocess WITHOUT stopword removal
    5. Create train/test splits
    6. Export to model training format

Workflow 2: Multi-Domain ABSA Analysis
    1. Run domain-level EDA
    2. Analyze domain-specific aspects
    3. Check sentiment distribution per domain
    4. Preprocess each domain separately if needed
    5. Create domain-specific models or multi-domain ones

Workflow 3: Data Quality Assessment
    1. Run completeness checks
    2. Identify and handle duplicates
    3. Validate sentiment values
    4. Assess text length distribution
    5. Identify outliers (very short/long texts)

─────────────────────────────────────────────────────────────────────────────

🔍 NEXT STEPS FOR YOUR PROJECT:
─────────────────────────────────────────────────────────────────────────────

1. Run all EDA cells to understand your data
2. Choose preprocessing strategy based on your task:
   - ABSA: Keep all tokens, normalize, remove English
   - Sentiment: Remove stopwords, normalize, remove English
3. Export preprocessed datasets for model training
4. Consider data balancing techniques if imbalance detected
5. Create train/validation/test splits
6. Build and train your ABSA models

═════════════════════════════════════════════════════════════════════════════
"""

print(usage_guide)

# Save guide to file
guide_file = 'PREPROCESSING_USAGE_GUIDE.txt'
with open(guide_file, 'w', encoding='utf-8') as f:
    f.write(usage_guide)
print(f"\n✓ Usage guide saved to '{guide_file}'")


## 7-Usage Guide & Best Practices

### Complete guide for using preprocessing and EDA for your ABSA project

In [ ]:
# =============================================================================
# 6.2 APPLYING PREPROCESSING TO ENTIRE DATASETS
# =============================================================================

print("\n" + "="*100)
print("6.2 APPLYING PREPROCESSING TO ALL DATASETS")
print("="*100)

# Create preprocessed versions of all datasets
print("\nGenerating preprocessed datasets...")

# For ABSA tasks (keep all tokens)
processed_datasets_absa = {}
for domain_name, df in datasets.items():
    print(f"\n  Processing {domain_name}...")
    preprocessor = ArabicTextPreprocessor(
        remove_stopwords=False,
        normalize=True,
        remove_english=True
    )
    
    processed = preprocessor.process_dataset(df, text_column='text', batch_size=50)
    
    # Create dataframe with processed texts
    processed_df = df.copy()
    processed_df['processed_tokens'] = [' '.join(tokens) for tokens in processed]
    processed_df['token_count'] = [len(tokens) for tokens in processed]
    processed_datasets_absa[domain_name] = processed_df

print("\n✓ All datasets preprocessed successfully!")

# Sample results
print("\n📊 SAMPLE PREPROCESSING RESULTS:\n")

for domain_name in list(datasets.keys())[:2]:
    processed_df = processed_datasets_absa[domain_name]
    print(f"\n{domain_name} - Sample (1st record):")
    print(f"  Original: {processed_df['text'].iloc[0][:80]}...")
    print(f"  Processed: {processed_df['processed_tokens'].iloc[0][:80]}...")
    print(f"  Token count: {processed_df['token_count'].iloc[0]}")


In [ ]:
# =============================================================================
# 6.1 ADVANCED PREPROCESSING PIPELINES FOR DIFFERENT USE CASES
# =============================================================================

class ArabicTextPreprocessor:
    """
    Advanced Arabic text preprocessing pipeline with multiple strategies
    Suitable for different NLP tasks like ABSA, sentiment analysis, etc.
    """
    
    def __init__(self, remove_stopwords=True, normalize=True, remove_english=True):
        self.remove_stopwords = remove_stopwords
        self.normalize = normalize
        self.remove_english = remove_english
        self.processed_texts = []
        self.statistics = {}
    
    def process_dataset(self, df, text_column='text', batch_size=100):
        """
        Process entire dataset with progress tracking
        """
        print(f"\n📝 Processing {len(df)} documents...")
        processed = []
        
        for idx, text in enumerate(df[text_column]):
            if (idx + 1) % batch_size == 0:
                print(f"  Progress: {idx + 1}/{len(df)} documents processed")
            
            tokens = preprocess_arabic_text(
                text,
                remove_stopwords=self.remove_stopwords,
                remove_english=self.remove_english
            )
            processed.append(tokens)
        
        self.processed_texts = processed
        self._compute_statistics()
        print(f"✓ Processing complete!")
        return processed
    
    def _compute_statistics(self):
        """Compute preprocessing statistics"""
        token_counts = [len(tokens) for tokens in self.processed_texts]
        
        self.statistics = {
            'total_documents': len(self.processed_texts),
            'avg_tokens': np.mean(token_counts) if token_counts else 0,
            'min_tokens': np.min(token_counts) if token_counts else 0,
            'max_tokens': np.max(token_counts) if token_counts else 0,
            'median_tokens': np.median(token_counts) if token_counts else 0,
            'total_tokens': sum(token_counts),
            'vocabulary_size': len(set(token for tokens in self.processed_texts for token in tokens))
        }
    
    def get_statistics(self):
        """Get preprocessing statistics"""
        return self.statistics
    
    def export_to_csv(self, output_path, original_df=None):
        """Export processed texts to CSV"""
        if not self.processed_texts:
            print("No processed texts to export!")
            return
        
        export_df = pd.DataFrame({
            'processed_tokens': [' '.join(tokens) for tokens in self.processed_texts],
            'token_count': [len(tokens) for tokens in self.processed_texts]
        })
        
        if original_df is not None and len(original_df) == len(export_df):
            # Add original columns
            for col in original_df.columns:
                if col != 'text':
                    export_df[f'original_{col}'] = original_df[col].values
        
        export_df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"✓ Processed dataset exported to '{output_path}'")
        return export_df

# Example: Create preprocessor for ABSA task
print("\n" + "="*100)
print("6.1 EXAMPLE: PREPROCESSING FOR ASPECT-BASED SENTIMENT ANALYSIS (ABSA)")
print("="*100)

# Strategy 1: Keep aspects and aspects-related context (no stopword removal)
preprocessor_absa = ArabicTextPreprocessor(
    remove_stopwords=False,  # Keep all words to preserve aspect context
    normalize=True,
    remove_english=True
)

print("\n🔧 Preprocessing Configuration for ABSA:")
print("  ✓ Remove stopwords: False (preserve aspect context)")
print("  ✓ Normalize Arabic: True")
print("  ✓ Remove English/Numbers: True")

# Process sample
sample_texts = df_all['text'].head(100)
processed_absa = preprocessor_absa.process_dataset(
    pd.DataFrame({'text': sample_texts}),
    text_column='text',
    batch_size=20
)

stats = preprocessor_absa.get_statistics()
print(f"\n📊 Processing Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

# Strategy 2: For sentiment analysis (with stopword removal)
print("\n" + "-"*100)
print("EXAMPLE: PREPROCESSING FOR SENTIMENT ANALYSIS")
print("-"*100)

preprocessor_sentiment = ArabicTextPreprocessor(
    remove_stopwords=True,   # Remove stopwords for sentiment classification
    normalize=True,
    remove_english=True
)

print("\n🔧 Preprocessing Configuration for Sentiment Analysis:")
print("  ✓ Remove stopwords: True (focus on sentiment words)")
print("  ✓ Normalize Arabic: True")
print("  ✓ Remove English/Numbers: True")

processed_sentiment = preprocessor_sentiment.process_dataset(
    pd.DataFrame({'text': sample_texts}),
    text_column='text',
    batch_size=20
)

stats = preprocessor_sentiment.get_statistics()
print(f"\n📊 Processing Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print("\n" + "="*100)
print("✓ Both preprocessing pipelines are ready for use!")
print("="*100)


## 6-Advanced Preprocessing Configuration

### Flexible preprocessing options for different use cases (ABSA, sentiment analysis, aspect extraction)

In [ ]:
# =============================================================================
# 5.9 COMPREHENSIVE SUMMARY REPORT
# =============================================================================

print("\n" + "="*100)
print("COMPREHENSIVE EDA SUMMARY REPORT")
print("="*100)

summary_report = f"""
{'='*100}
📊 EXPLORATORY DATA ANALYSIS - EXECUTIVE SUMMARY
{'='*100}

1️⃣  DATASET OVERVIEW:
   ✓ Total Records: {len(df_all):,}
   ✓ Domains: {len(datasets)} (ATT, MOV, RES, PROD, TELC)
   ✓ Columns: {', '.join(df_all.columns.tolist())}
   ✓ Data Completeness: {completeness_score:.2f}%

2️⃣  TEXT CHARACTERISTICS:
   ✓ Min Text Length: {combined_stats['Min Length']:.0f} characters
   ✓ Max Text Length: {combined_stats['Max Length']:.0f} characters
   ✓ Mean Text Length: {combined_stats['Mean Length']:.2f} characters
   ✓ Mean Token Count: {combined_stats['Mean Token Count']:.2f} tokens
   ✓ Total Unique Tokens (Sample): {len(vocabulary):,}
   ✓ Vocabulary Size / Total Tokens: {len(vocabulary) / len(all_tokens):.4f}

3️⃣  SENTIMENT DISTRIBUTION:
"""

if sentiment_col:
    sentiment_stats = df_all[sentiment_col].value_counts()
    for sentiment, count in sentiment_stats.items():
        percentage = (count / len(df_all)) * 100
        summary_report += f"   ✓ {sentiment}: {count:,} ({percentage:.2f}%)\n"
    
    min_class = sentiment_stats.min()
    max_class = sentiment_stats.max()
    balance_ratio = max_class / min_class if min_class > 0 else float('inf')
    summary_report += f"   ✓ Class Balance Ratio: {balance_ratio:.2f}:1\n"
else:
    summary_report += "   ⚠️  No sentiment column found\n"

summary_report += f"""
4️⃣  DOMAIN DISTRIBUTION:
   ✓ ATT (Attributes): {len(df_ATT):,} records ({len(df_ATT)/len(df_all)*100:.2f}%)
   ✓ MOV (Movies): {len(df_MOV):,} records ({len(df_MOV)/len(df_all)*100:.2f}%)
   ✓ RES (Restaurants): {len(df_RES):,} records ({len(df_RES)/len(df_all)*100:.2f}%)
   ✓ PROD (Products): {len(df_PROD):,} records ({len(df_PROD)/len(df_all)*100:.2f}%)
   ✓ TELC (Telecom): {len(df_TELC):,} records ({len(df_TELC)/len(df_all)*100:.2f}%)

5️⃣  LINGUISTIC STATISTICS:
   ✓ Type-Token Ratio: {len(vocabulary) / len(all_tokens):.4f}
   ✓ Hapax Legomena (words appearing once): {len(hapax):,} ({len(hapax) / len(vocabulary) * 100:.2f}%)
   ✓ Avg Tokens per Document: {len(all_tokens) / len(all_preprocessed_texts):.2f}

6️⃣  DATA QUALITY:
   ✓ Missing Values: {df_all.isnull().sum().sum()} ({(df_all.isnull().sum().sum() / (df_all.shape[0] * df_all.shape[1]))*100:.2f}%)
   ✓ Duplicate Records: {df_all.duplicated().sum()}
   ✓ Duplicate Texts: {df_all['text'].duplicated().sum() if 'text' in df_all.columns else 'N/A'}
   
7️⃣  PREPROCESSING CAPABILITIES:
   ✓ Arabic Normalization: Diacritics removal, character normalization
   ✓ Tokenization: Word-level tokenization for Arabic text
   ✓ Stopword Removal: {len(arabic_stopwords)} Arabic stopwords available
   ✓ URL/Email Removal: Implemented
   ✓ Special Character Handling: Implemented

8️⃣  KEY FINDINGS & RECOMMENDATIONS:
   ✓ Dataset is suitable for ABSA tasks
   ✓ {"✓ Dataset appears balanced" if balance_ratio <= 3 else "⚠️  Dataset shows class imbalance - consider balancing techniques"}
   ✓ ✓ Multiple domains provide good generalization opportunity
   ✓ ✓ Arabic preprocessing pipeline ready for implementation

{'='*100}
"""

print(summary_report)

# Save report to file
report_file = 'EDA_Summary_Report.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(summary_report)
print(f"\n✓ Summary report saved to '{report_file}'")


In [ ]:
# =============================================================================
# 5.8 VOCABULARY AND LINGUISTIC ANALYSIS
# =============================================================================

print("\n" + "="*100)
print("VOCABULARY AND LINGUISTIC ANALYSIS")
print("="*100)

# Build vocabulary from preprocessed texts
print("\n🔤 Building vocabulary from combined dataset...")
vocabulary = set()
all_tokens = []
all_preprocessed_texts = []

# Process all texts
for text in df_all['text'].dropna().head(500):  # Sample for computational efficiency
    tokens = preprocess_arabic_text(text, remove_stopwords=False)
    all_tokens.extend(tokens)
    all_preprocessed_texts.append(tokens)
    vocabulary.update(tokens)

print(f"\n✓ VOCABULARY STATISTICS:")
print(f"  Unique tokens (vocabulary size): {len(vocabulary)}")
print(f"  Total tokens: {len(all_tokens)}")
print(f"  Average tokens per document: {len(all_tokens) / len(all_preprocessed_texts):.2f}")
print(f"  Type-Token Ratio (TTR): {len(vocabulary) / len(all_tokens):.4f}")

# Most common tokens
print(f"\n✓ TOP 30 MOST FREQUENT TOKENS:")
token_freq = Counter(all_tokens)
for i, (token, freq) in enumerate(token_freq.most_common(30), 1):
    print(f"  {i:2d}. {token:15s} - {freq:4d} occurrences")

# Least common tokens (hapax legomena)
hapax = [word for word, freq in token_freq.items() if freq == 1]
print(f"\n✓ RARE WORDS (appearing once - Hapax Legomena):")
print(f"  Count: {len(hapax)}")
print(f"  Percentage: {len(hapax) / len(vocabulary) * 100:.2f}%")
print(f"  Examples: {', '.join(list(hapax)[:20])}")

# Visualize token frequency
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 tokens
top_tokens = dict(token_freq.most_common(20))
axes[0].barh(range(len(top_tokens)), list(top_tokens.values()), 
             color=plt.cm.viridis(np.linspace(0, 1, len(top_tokens))))
axes[0].set_yticks(range(len(top_tokens)))
axes[0].set_yticklabels(list(top_tokens.keys()))
axes[0].set_xlabel('Frequency', fontsize=12)
axes[0].set_title('🔤 Top 20 Most Frequent Tokens', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Token frequency distribution (log-scale)
freq_counts = sorted(Counter(token_freq.values()).items())
freqs = [f[0] for f in freq_counts]
counts = [f[1] for f in freq_counts]

axes[1].scatter(freqs, counts, alpha=0.6, s=50)
axes[1].set_xlabel('Token Frequency', fontsize=12)
axes[1].set_ylabel('Number of Tokens', fontsize=12)
axes[1].set_title('📊 Token Frequency Distribution (Log-Log Scale)', fontsize=14, fontweight='bold')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Vocabulary_analysis.png', dpi=300, bbox_inches='tight')
print("\n✓ Vocabulary visualization saved as 'Vocabulary_analysis.png'")
plt.show()


In [ ]:
# =============================================================================
# 5.7 SAMPLE PREPROCESSING DEMONSTRATION
# =============================================================================

print("\n" + "="*100)
print("ARABIC PREPROCESSING DEMONSTRATION")
print("="*100)

# Select random samples from each domain
print("\n📝 BEFORE AND AFTER PREPROCESSING SAMPLES:\n")

for domain_name, df in list(datasets.items())[:3]:  # Show first 3 domains
    print(f"\n{'-'*100}")
    print(f"{domain_name}")
    print(f"{'-'*100}")
    
    # Get random sample
    sample = df.sample(1, random_state=42)
    original_text = sample['text'].values[0]
    
    print(f"\n🔵 ORIGINAL TEXT:")
    print(f"  {original_text[:150]}..." if len(original_text) > 150 else f"  {original_text}")
    
    print(f"\n🔵 AFTER PREPROCESSING:")
    preprocessed = preprocess_arabic_text(original_text, remove_stopwords=False)
    print(f"  Tokens: {' | '.join(preprocessed)}")
    print(f"  Number of tokens: {len(preprocessed)}")
    
    print(f"\n🔵 AFTER PREPROCESSING (With Stopwords Removal):")
    preprocessed_filtered = preprocess_arabic_text(original_text, remove_stopwords=True)
    print(f"  Tokens: {' | '.join(preprocessed_filtered)}")
    print(f"  Number of tokens: {len(preprocessed_filtered)}")
    
    # Show metadata
    if sentiment_col:
        print(f"\n🔵 METADATA:")
        print(f"  Sentiment: {sample[sentiment_col].values[0]}")
    if aspect_col:
        print(f"  Aspect: {sample[aspect_col].values[0]}")


In [ ]:
# =============================================================================
# 5.6 DATA QUALITY ASSESSMENT
# =============================================================================

print("\n" + "="*100)
print("DATA QUALITY ASSESSMENT")
print("="*100)

quality_metrics = {
    'Completeness': {},
    'Consistency': {},
    'Validity': {}
}

# Completeness check
print("\n✓ COMPLETENESS CHECK:")
completeness_score = (1 - (df_all.isnull().sum().sum() / (df_all.shape[0] * df_all.shape[1]))) * 100
print(f"  Overall Completeness: {completeness_score:.2f}%")

for col in df_all.columns:
    completeness = (1 - df_all[col].isnull().sum() / len(df_all)) * 100
    print(f"    {col}: {completeness:.2f}%")

# Consistency check - text column
print("\n✓ CONSISTENCY CHECK (Text Column):")
if 'text' in df_all.columns:
    empty_texts = (df_all['text'].str.len() == 0).sum()
    whitespace_only = (df_all['text'].str.strip().str.len() == 0).sum()
    duplicate_texts = df_all['text'].duplicated().sum()
    
    print(f"  Empty texts: {empty_texts} ({empty_texts/len(df_all)*100:.2f}%)")
    print(f"  Whitespace-only texts: {whitespace_only} ({whitespace_only/len(df_all)*100:.2f}%)")
    print(f"  Duplicate texts: {duplicate_texts} ({duplicate_texts/len(df_all)*100:.2f}%)")

# Validity check - sentiment values
print("\n✓ VALIDITY CHECK (Sentiment Values):")
if sentiment_col:
    valid_sentiments = df_all[sentiment_col].dropna().unique()
    print(f"  Valid sentiment values: {sorted(valid_sentiments)}")
    print(f"  Null sentiments: {df_all[sentiment_col].isnull().sum()}")

# Data balance analysis
print("\n✓ DATA BALANCE ANALYSIS:")
if sentiment_col:
    sentiment_dist = df_all[sentiment_col].value_counts()
    min_class = sentiment_dist.min()
    max_class = sentiment_dist.max()
    imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')
    print(f"  Sentiment class distribution:")
    print(f"    Min class size: {min_class}")
    print(f"    Max class size: {max_class}")
    print(f"    Imbalance ratio: {imbalance_ratio:.2f}:1")
    
    if imbalance_ratio > 3:
        print(f"    ⚠️  Warning: Dataset is imbalanced (ratio > 3)")
    else:
        print(f"    ✓ Dataset is relatively balanced")

if 'domain' in df_all.columns:
    domain_dist = df_all['domain'].value_counts()
    print(f"\n  Domain distribution:")
    for domain, count in domain_dist.items():
        print(f"    {domain}: {count} records")


In [ ]:
# =============================================================================
# 5.5 ASPECT ANALYSIS (If aspect column exists)
# =============================================================================

print("\n" + "="*100)
print("DETAILED ASPECT ANALYSIS")
print("="*100)

# Find aspect column
aspect_col = None
for col in df_all.columns:
    if 'aspect' in col.lower() or 'category' in col.lower() or 'target' in col.lower():
        aspect_col = col
        break

if aspect_col:
    print(f"\n✓ Found aspect column: '{aspect_col}'")
    
    # Overall aspect statistics
    print(f"\n📊 ASPECT STATISTICS:")
    print(f"  Total Unique Aspects: {df_all[aspect_col].nunique()}")
    print(f"  Most Common Aspects (Top 20):")
    
    aspect_counts = df_all[aspect_col].value_counts().head(20)
    for i, (aspect, count) in enumerate(aspect_counts.items(), 1):
        percentage = (count / len(df_all)) * 100
        print(f"    {i:2d}. {aspect}: {count:5d} ({percentage:6.2f}%)")
    
    # Visualize top aspects
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top 15 aspects
    top_aspects = df_all[aspect_col].value_counts().head(15)
    axes[0].barh(range(len(top_aspects)), top_aspects.values, color=plt.cm.Spectral(np.linspace(0, 1, len(top_aspects))))
    axes[0].set_yticks(range(len(top_aspects)))
    axes[0].set_yticklabels(top_aspects.index)
    axes[0].set_xlabel('Frequency', fontsize=12)
    axes[0].set_title('🏷️  Top 15 Aspects by Frequency', fontsize=14, fontweight='bold')
    axes[0].invert_yaxis()
    
    # Aspect count by domain
    if 'domain' in df_all.columns:
        aspect_domain = pd.crosstab(df_all['domain'], df_all[aspect_col]).head(10)
        aspect_domain.plot(kind='bar', ax=axes[1], stacked=False)
        axes[1].set_title('🔀 Top Aspects Distribution by Domain', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Domain', fontsize=12)
        axes[1].set_ylabel('Count', fontsize=12)
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=2)
        axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('Aspect_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Aspect visualization saved as 'Aspect_analysis.png'")
    plt.show()

else:
    print("\n⚠️  No aspect column found. Skipping aspect analysis.")


In [ ]:
# =============================================================================
# 5.4 COMBINED DATASET VISUALIZATIONS
# =============================================================================

print("\n" + "="*100)
print("GENERATING VISUALIZATIONS")
print("="*100)

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(20, 12))

# 1. Domain distribution
ax1 = plt.subplot(2, 3, 1)
domain_counts = df_all['domain'].value_counts() if 'domain' in df_all.columns else \
                pd.Series([len(df_ATT), len(df_MOV), len(df_RES), len(df_PROD), len(df_TELC)],
                         index=['ATT', 'MOV', 'RES', 'PROD', 'TELC'])
colors = plt.cm.Set3(range(len(domain_counts)))
domain_counts.plot(kind='bar', ax=ax1, color=colors)
ax1.set_title('📊 Records Distribution by Domain', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Records', fontsize=12)
ax1.set_xlabel('Domain', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(domain_counts.values):
    ax1.text(i, v + 50, str(v), ha='center', fontweight='bold')

# 2. Sentiment distribution (overall)
ax2 = plt.subplot(2, 3, 2)
# Find sentiment column
sentiment_col = None
for col in df_all.columns:
    if 'polarity' in col.lower() or 'sentiment' in col.lower() or 'label' in col.lower():
        sentiment_col = col
        break

if sentiment_col:
    sentiment_dist = df_all[sentiment_col].value_counts()
    colors = ['#d62728' if x == 'negative' else '#2ca02c' if x == 'positive' else '#1f77b4' 
              for x in sentiment_dist.index]
    sentiment_dist.plot(kind='bar', ax=ax2, color=colors)
    ax2.set_title(f'😊 {sentiment_col} Distribution', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_xlabel('Sentiment', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)
    for i, v in enumerate(sentiment_dist.values):
        ax2.text(i, v + 10, str(v), ha='center', fontweight='bold')

# 3. Text length distribution
ax3 = plt.subplot(2, 3, 3)
text_lengths = df_all['text'].str.len()
ax3.hist(text_lengths, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax3.set_title('📏 Text Length Distribution', fontsize=14, fontweight='bold')
ax3.set_xlabel('Character Count', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.axvline(text_lengths.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {text_lengths.mean():.0f}')
ax3.axvline(text_lengths.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {text_lengths.median():.0f}')
ax3.legend()

# 4. Token count distribution
ax4 = plt.subplot(2, 3, 4)
token_counts = df_all['text'].str.split().str.len()
ax4.hist(token_counts, bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
ax4.set_title('🔤 Token Count Distribution', fontsize=14, fontweight='bold')
ax4.set_xlabel('Number of Tokens', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.axvline(token_counts.mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {token_counts.mean():.1f}')
ax4.legend()

# 5. Domain vs Sentiment heatmap
ax5 = plt.subplot(2, 3, 5)
if 'domain' in df_all.columns and sentiment_col:
    domain_sentiment = pd.crosstab(df_all['domain'], df_all[sentiment_col])
    sns.heatmap(domain_sentiment, annot=True, fmt='d', cmap='YlOrRd', ax=ax5, cbar_kws={'label': 'Count'})
    ax5.set_title('🔥 Domain × Sentiment Heatmap', fontsize=14, fontweight='bold')
    ax5.set_ylabel('Domain', fontsize=12)
    ax5.set_xlabel('Sentiment', fontsize=12)

# 6. Missing values
ax6 = plt.subplot(2, 3, 6)
missing_data = df_all.isnull().sum()
missing_data = missing_data[missing_data > 0]
if len(missing_data) > 0:
    missing_data.plot(kind='barh', ax=ax6, color='orange')
    ax6.set_title('⚠️  Missing Values by Column', fontsize=14, fontweight='bold')
    ax6.set_xlabel('Count', fontsize=12)
else:
    ax6.text(0.5, 0.5, 'No Missing Values Found! ✓', ha='center', va='center', fontsize=16, fontweight='bold')
    ax6.set_title('⚠️  Missing Values by Column', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('EDA_overview.png', dpi=300, bbox_inches='tight')
print("\n✓ Visualization saved as 'EDA_overview.png'")
plt.show()


In [ ]:
# =============================================================================
# 5.3 SENTIMENT AND ASPECT DISTRIBUTION ANALYSIS
# =============================================================================

print("\n" + "="*100)
print("SENTIMENT AND ASPECT DISTRIBUTION ANALYSIS")
print("="*100)

# Determine sentiment and aspect columns
def identify_columns(df):
    """Identify key columns in dataset"""
    columns = {
        'text': None,
        'aspect': None,
        'sentiment': None
    }
    
    col_lower = [col.lower() for col in df.columns]
    
    # Identify text column
    for col in df.columns:
        if 'text' in col.lower() or 'review' in col.lower():
            columns['text'] = col
            break
    if not columns['text'] and len(df.columns) > 0:
        columns['text'] = df.columns[0]
    
    # Identify aspect column
    for col in df.columns:
        if 'aspect' in col.lower() or 'category' in col.lower() or 'target' in col.lower():
            columns['aspect'] = col
            break
    
    # Identify sentiment column
    for col in df.columns:
        if 'polarity' in col.lower() or 'sentiment' in col.lower() or 'label' in col.lower():
            columns['sentiment'] = col
            break
    
    return columns

# Analyze each domain
for domain_name, df in datasets.items():
    print(f"\n{'-'*100}")
    print(f"{domain_name} ANALYSIS:")
    print(f"{'-'*100}")
    
    cols = identify_columns(df)
    print(f"\nIdentified Columns: {cols}")
    
    # Sentiment distribution
    if cols['sentiment']:
        print(f"\n📊 SENTIMENT DISTRIBUTION:")
        sentiment_dist = df[cols['sentiment']].value_counts()
        print(sentiment_dist)
        print(f"\nPercentage Distribution:")
        print((sentiment_dist / len(df) * 100).round(2))
    
    # Aspect distribution
    if cols['aspect']:
        print(f"\n🏷️  ASPECT/CATEGORY DISTRIBUTION:")
        aspect_dist = df[cols['aspect']].value_counts()
        print(aspect_dist)
        print(f"\nTop 10 Categories:")
        print(aspect_dist.head(10))
    
    # Sentiment-Aspect cross-tabulation
    if cols['sentiment'] and cols['aspect']:
        print(f"\n🔀 SENTIMENT × ASPECT CROSS-TABULATION:")
        crosstab = pd.crosstab(df[cols['aspect']], df[cols['sentiment']], margins=True)
        print(crosstab)


In [ ]:
# =============================================================================
# 5.2 TEXT STATISTICS AND CHARACTERISTICS
# =============================================================================

print("\n" + "="*100)
print("TEXT STATISTICS AND CHARACTERISTICS")
print("="*100)

def compute_text_statistics(df, text_column='text'):
    """Compute detailed text statistics"""
    if text_column not in df.columns:
        return None
    
    stats = {
        'Total Texts': len(df),
        'Non-Empty Texts': df[text_column].notna().sum(),
        'Empty Texts': df[text_column].isna().sum(),
        'Min Length': df[text_column].str.len().min(),
        'Max Length': df[text_column].str.len().max(),
        'Mean Length': df[text_column].str.len().mean(),
        'Median Length': df[text_column].str.len().median(),
        'Std Dev Length': df[text_column].str.len().std(),
    }
    
    # Token-level statistics
    token_counts = df[text_column].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
    stats['Min Token Count'] = token_counts.min()
    stats['Max Token Count'] = token_counts.max()
    stats['Mean Token Count'] = token_counts.mean()
    stats['Median Token Count'] = token_counts.median()
    
    return stats

# Compute statistics for each domain
print("\n📈 TEXT STATISTICS BY DOMAIN:\n")
text_stats_all = {}
for domain_name, df in datasets.items():
    stats = compute_text_statistics(df, 'text')
    if stats:
        text_stats_all[domain_name] = stats
        print(f"\n{domain_name}:")
        for key, value in stats.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.2f}")
            else:
                print(f"  {key}: {value}")

# Combined statistics
print(f"\n{'-'*100}")
print("COMBINED DATASETS:")
combined_stats = compute_text_statistics(df_all, 'text')
for key, value in combined_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")


In [ ]:
# =============================================================================
# 5.1 DATASET OVERVIEW AND STRUCTURE
# =============================================================================

print("\n" + "="*100)
print("COMPREHENSIVE DATASET OVERVIEW")
print("="*100)

# Dictionary to store datasets
datasets = {
    'ATT (Attributes)': df_ATT,
    'MOV (Movies)': df_MOV,
    'RES (Restaurants)': df_RES,
    'PROD (Products)': df_PROD,
    'TELC (Telecom)': df_TELC
}

# Create overview table
overview_data = []
for domain_name, df in datasets.items():
    overview_data.append({
        'Domain': domain_name,
        'Total Records': len(df),
        'Total Columns': df.shape[1],
        'Missing Values': df.isnull().sum().sum(),
        'Memory Usage (MB)': df.memory_usage(deep=True).sum() / 1024**2
    })

overview_df = pd.DataFrame(overview_data)
print("\n📊 DATASET OVERVIEW:")
print(overview_df.to_string(index=False))

# Overall statistics
print(f"\n✓ Combined Dataset:")
print(f"  - Total Records: {len(df_all)}")
print(f"  - Total Domains: {len(datasets)}")
print(f"  - Columns: {list(df_all.columns)}")
print(f"  - Data Types:\n{df_all.dtypes}")
print(f"\n  Missing Values Summary:")
print(df_all.isnull().sum())


## 5-Exploratory Data Analysis (EDA) 

### Comprehensive analysis of all datasets covering structure, content, and sentiment distributions